# GGIT gas pipelines summary sheets — November 2026 release

Produces the gas-pipeline summary tables (km by region, country, owner, start year; cost
per km; capex) for the November 2026 GGIT data release. Writes one Excel workbook that is
pasted into the shared summary tables:
https://docs.google.com/spreadsheets/d/1NbEpGt2K5nY0XTSB_vlOyw9Ug8ZmvvOaRPuO9TgISIw/edit

**Method is documented in `../2026-q2-oil-pipelines/README.md` — read that first.** This
notebook is a fork of the one validated there. `README.md` in this folder records only what
differs, and two things do:

1. **Reads go through `gem-db-ops` / the `gws` CLI, not `pygsheets`.** The `gem-analysis`
   service account was deleted on 2026-07-31, so `pygsheets.authorize` cannot work.
2. **The capex tabs are fuel-filtered.** Published GGIT gas capex tabs through the Nov 2025
   release summed *every* fuel — oil, NGL, LPG, CO2, naphtha — into the "gas pipelines"
   capex table. This notebook counts gas only, so its capex tabs are **not comparable**
   with previously published ones. See `README.md`.

## imports and configuration

In [ ]:
%pip install -q -e ../../../gem-tracker-constants

import datetime
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Canonical fuel buckets, status orderings, and the gas-and-hydrogen collapse helper.
# Source of truth: the in-repo gem-tracker-constants package (installed editable above).
from gem_tracker_constants import (
    GAS_FUEL_OPTIONS,
    OIL_FUEL_OPTIONS,
    NGL_FUEL_OPTIONS,
    PIPELINE_STATUS as STATUS_LIST,
    PIPELINE_EXCEL_STATUS as EXCEL_STATUS_LIST,
    PIPELINE_IN_DEV_COL as IN_DEV_COL,
    collapse_gas_and_hydrogen,
)

# Sheets reads come from the sibling gem-db-ops repo — the single source of truth for
# pulling GEM data. It wraps the read-only `gws` profile (~/.config/gws-gem); there is no
# service-account path any more.
GEM_DB_OPS = (Path.cwd() / "../../../../gem-db-ops").resolve()
assert GEM_DB_OPS.is_dir(), f"gem-db-ops not found at {GEM_DB_OPS}"
sys.path.insert(0, str(GEM_DB_OPS))
import gem_sheets

In [ ]:
# === config ===========================================================
# Set FUEL_TYPE to one of: "Gas", "Oil", "NGL". Run the notebook once per fuel.
FUEL_TYPE = "Gas"

# Source spreadsheet.
# Past release snapshots, for reference:
#   1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA  — Dec 2023 GGIT gas
#   1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs  — Dec 2024 GGIT gas
#   1xjaeq0OwdN-Orht7Q7ynPHB2uY_gnMvAeha-QHkzoZw  — Nov 2025 GGIT gas
#   1gChRPYLrcirx3lNI_DHWs5GaKTtfXgELVg1bHix8zqs  — June 2026 GOIT oil/NGL
ROLLING_BACKEND_KEY = "1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek"  # CURRENT, live

# TODO before the release run: repoint to the Nov 2026 snapshot once it is cut.
SPREADSHEET_KEY = ROLLING_BACKEND_KEY

# Where to write the Excel summary. Defaults to alongside this notebook.
OUTPUT_DIR = Path.cwd()

# Region filter:
#   "Global"              — every country
#   "AsiaGasTracker"      — countries flagged AsiaGasTracker == "Yes"
#   "EuroGasTracker"      — countries flagged EuroGasTracker == "Yes"
#   "AfricaGasTracker"    — countries flagged AfricaGasTracker == "Yes"
#   "LatinAmericaTracker" — countries flagged LatinAmericaTracker == "Yes"
REGION_NAME = "Global"

if SPREADSHEET_KEY == ROLLING_BACKEND_KEY:
    print(
        "!" * 78,
        "WARNING: reading the ROLLING live backend, not a frozen release snapshot.",
        "Fine for a preview; NOT reproducible. Repoint SPREADSHEET_KEY at the",
        "Nov 2026 snapshot before generating tables for publication.",
        "!" * 78,
        sep="\n",
    )

In [ ]:
# Fuel buckets are imported above from gem-tracker-constants.
# Per-run glue: pick the sheet to read, the bucket to filter on, and the output label.
FUEL_CONFIG = {
    "Gas": {"sheet": "Gas pipelines",     "options": GAS_FUEL_OPTIONS, "label": "Gas"},
    "Oil": {"sheet": "Oil/NGL pipelines", "options": OIL_FUEL_OPTIONS, "label": "Oil"},
    "NGL": {"sheet": "Oil/NGL pipelines", "options": NGL_FUEL_OPTIONS, "label": "NGL"},
}
assert FUEL_TYPE in FUEL_CONFIG, f"unknown FUEL_TYPE {FUEL_TYPE!r}"
FUEL_OPTIONS = FUEL_CONFIG[FUEL_TYPE]["options"]
FUEL_LABEL = FUEL_CONFIG[FUEL_TYPE]["label"]

In [ ]:
# Status orderings (STATUS_LIST, EXCEL_STATUS_LIST, IN_DEV_COL) are imported above
# from gem-tracker-constants.

# Columns that should be numeric. The Sheets API returns everything as strings, so we
# coerce at load time (a single source of truth) — this prevents the entire class of bug
# where an empty string slips past .notna() and crashes .quantile() / sort().
NUMERIC_COLS_PIPES = [
    "LengthMergedKm",
    "StartYearEarliest",
    "ProposalYear",
    "ConstructionYear",
    "ShelvedYear",
    "CancelledYear",
    "CostUSDPerKm",
    "CapacityBcm/y",
    "DiameterInches",
]
NUMERIC_COLS_RATIOS = [
    "LengthMergedKmByCountry",
    "LengthEstimateKmByCountry",  # may contain comma-formatted strings like "1,236.43"
    "LengthKnownKmByCountry",     # used by the cost-per-km estimate
    "LengthPerCountryFraction",
    "CostUSDPerKm",
]

# Columns the backend has renamed across releases. Resolved to the first name present,
# then aliased to the canonical name the rest of the notebook uses.
COLUMN_ALIASES = {
    # canonical            candidates, in preference order
    "CountriesOrAreas": ["CountriesOrAreas", "Countries"],
}

## load the source workbook

Read through `gem_sheets.read_tab_values(title, sheet_key)` — the explicit two-argument
form. Do **not** route these through the `gem_sheets.py` CLI: `--sheet-key` is silently
ignored for its four registered tabs (`Gas pipelines` among them), which sends the read to
the rolling backend no matter which snapshot you asked for.

Header rows drift as banner rows are added and removed from the backend (the ratios tab
gained an A1 run-stamp row on 2026-08-05; the country dictionary lost its banner on
2026-08-02). Rather than pin an offset per release, `read_tab` finds the header by looking
for a row that contains a set of known column names, and fails loudly if it cannot.

In [ ]:
def _dedupe_header(header: list[str]) -> list[str]:
    """Make a header row usable as DataFrame columns: name the blanks, suffix duplicates."""
    out, seen = [], {}
    for i, raw in enumerate(header):
        name = str(raw).strip() or f"Unnamed: {i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}.{seen[name]}"
        else:
            seen[name] = 0
        out.append(name)
    return out


def read_tab(title: str, required: set[str], sheet_key: str = None,
             max_header_row: int = 8) -> pd.DataFrame:
    """Read one worksheet into an all-string DataFrame, blanks as "".

    `required` is a set of column names used to locate the header row, so a banner row
    appearing or disappearing upstream does not silently shift every column.
    """
    values = gem_sheets.read_tab_values(title, sheet_key or SPREADSHEET_KEY)

    header_row = None
    for i, row in enumerate(values[:max_header_row]):
        if required <= {str(c).strip() for c in row}:
            header_row = i
            break
    if header_row is None:
        raise KeyError(
            f"could not find a header row in the first {max_header_row} rows of {title!r}: "
            f"no row contains all of {sorted(required)}. Check the tab against "
            f"SPREADSHEET_KEY — a column may have been renamed."
        )

    columns = _dedupe_header(values[header_row])
    width = len(columns)
    rows = [(list(r) + [""] * width)[:width] for r in values[header_row + 1:]]
    df = pd.DataFrame(rows, columns=columns, dtype=object)
    df = df.loc[~(df.map(lambda v: str(v).strip() == "")).all(axis=1)]  # drop blank rows
    print(f"  {title!r}: header at row {header_row}, {len(df):,} data rows x {width} cols")
    return df.reset_index(drop=True)


def apply_aliases(df: pd.DataFrame) -> pd.DataFrame:
    """Rename release-to-release column renames onto their canonical names."""
    df = df.copy()
    for canonical, candidates in COLUMN_ALIASES.items():
        present = [c for c in candidates if c in df.columns]
        if not present:
            continue
        if present[0] != canonical:
            df = df.rename(columns={present[0]: canonical})
    return df


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert listed columns to numeric. Strips thousands separators (commas) first,
    then `pd.to_numeric(errors="coerce")` turns anything unparseable into NaN.
    Columns absent from `df` are silently skipped."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        s = df[col]
        if s.dtype == object:
            s = s.astype(str).str.replace(",", "", regex=False)
        df[col] = pd.to_numeric(s, errors="coerce")
    return df

In [ ]:
print(f"reading {SPREADSHEET_KEY}")
pipes_sheet_title = FUEL_CONFIG[FUEL_TYPE]["sheet"]

pipes_df_orig = read_tab(
    pipes_sheet_title,
    required={"ProjectID", "PipelineName", "Status", "Fuel", "LengthMergedKm"},
).drop(columns="WKTFormat", errors="ignore")

country_ratios_df = read_tab(
    "Country ratios by pipeline",
    required={"ProjectID", "Country", "Status", "Fuel", "LengthMergedKmByCountry"},
).drop(columns="WKTFormat", errors="ignore")

region_df_orig = read_tab(
    "Country dictionary",
    required={"Country", "Region", "SubRegion"},
)

pipes_df_orig = apply_aliases(pipes_df_orig)
country_ratios_df = apply_aliases(country_ratios_df)

# coerce numeric columns at load time — single source of truth for type cleanup
pipes_df_orig = coerce_numeric(pipes_df_orig, NUMERIC_COLS_PIPES)
country_ratios_df = coerce_numeric(country_ratios_df, NUMERIC_COLS_RATIOS)

# fail loudly if a column the rest of the notebook needs went missing upstream
for name, df, needed in [
    ("pipes", pipes_df_orig, ["CountriesOrAreas", "RouteAccuracy", "CostUSDPerKm", "Wiki"]),
    ("ratios", country_ratios_df, ["Region", "SubRegion", "LengthKnownKmByCountry",
                                   "CostUSDPerKm", "Parent"]),
]:
    missing = [c for c in needed if c not in df.columns]
    assert not missing, f"{name} tab is missing {missing}"
print("all required columns present")

## clean inputs

In [ ]:
def clean_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """Replace `--` and empty-string sentinels with NaN."""
    return df.replace({"--": np.nan, "": np.nan}).infer_objects(copy=False)


# Same row filters as the release-downloads export
# (releases/downloads/convert-ggit-goit-to-tracker-release-downloads.ipynb,
# load_pipeline_data), so these summaries calculate on exactly the rows shipped
# in the data files.
pipes_df_orig = pipes_df_orig.loc[
    (pipes_df_orig["Status"] != "N/A")
    & (pipes_df_orig["PipelineName"] != "")
    & (pipes_df_orig["RouteAccuracy"] != "")
].copy()

# subset pipes to the chosen fuel bucket; collapse "Gas and Hydrogen" -> "Gas" for gas runs
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig["Fuel"].isin(FUEL_OPTIONS)].copy()
if FUEL_TYPE == "Gas":
    collapse_gas_and_hydrogen(pipes_df_orig)
    collapse_gas_and_hydrogen(country_ratios_df)

# the ratios tab needs the same status cleanup: rows with statuses outside
# STATUS_LIST ("N/A", "mixed status") fall out of the status pivots anyway,
# but the cost-per-km averages don't pivot by status and would otherwise
# include them
country_ratios_df = country_ratios_df.loc[
    country_ratios_df["Status"].isin(STATUS_LIST)
].copy()

# no Wiki filter on the ratios tab: pipelines without wiki pages still ship in
# the release downloads, so they count in the summaries too
country_ratios_df = clean_placeholders(country_ratios_df)

pipes_df_orig = clean_placeholders(pipes_df_orig)

print(f"{len(pipes_df_orig):,} {FUEL_LABEL} pipeline rows, "
      f"{len(country_ratios_df):,} ratio rows (all fuels)")

## region selection and country / region subsets

`REGION_NAME` filters the per-country and per-region tables. The cost-per-km means stay
global — see the cost section.

In [ ]:
if REGION_NAME == "Global":
    region_df_touse = region_df_orig
else:
    if REGION_NAME not in region_df_orig.columns:
        raise KeyError(
            f"REGION_NAME={REGION_NAME!r} but region_df_orig has no column {REGION_NAME!r}; "
            f"available: {[c for c in region_df_orig.columns if c.endswith('Tracker')]}"
        )
    region_df_touse = region_df_orig.loc[region_df_orig[REGION_NAME] == "Yes"]

region_df_touse_cleaned = region_df_touse.loc[
    (region_df_touse["Region"] != "--") & (region_df_touse["SubRegion"] != "--")
]
multiindex_region_subregion = (
    region_df_touse_cleaned.groupby(["Region", "SubRegion"])["Country"].count().index
)

# isin() against an explicit set — no regex, no substring false-positives
countries_in_region = set(region_df_touse["Country"])

country_ratios_df_touse = country_ratios_df.loc[
    country_ratios_df["Country"].isin(countries_in_region)
].copy()
pipes_df_touse = pipes_df_orig.loc[
    pipes_df_orig["CountriesOrAreas"]
    .fillna("")
    .str.split(", ")
    .map(lambda cs: any(c in countries_in_region for c in cs))
].copy()

print(f"region={REGION_NAME!r}: {len(countries_in_region)} countries, "
      f"{len(pipes_df_touse)} pipeline rows, {len(country_ratios_df_touse)} ratio rows")
multiindex_region_subregion

## set up Excel writer

In [ ]:
today = datetime.date.today().isoformat()
output_path = OUTPUT_DIR / f"GGIT-Summary-Sheets-{FUEL_LABEL}-{today}.xlsx"
print(f"writing to {output_path}")
excel_writer = pd.ExcelWriter(output_path)

## km by region and km by country

In [ ]:
country_ratios_df_subset = country_ratios_df_touse.loc[
    country_ratios_df_touse["Fuel"].isin(FUEL_OPTIONS)
]

country_list = sorted(country_ratios_df_subset["Country"].dropna().unique())


def pivot_km(group_cols, index) -> pd.DataFrame:
    """Sum LengthMergedKmByCountry by Status × group_cols, reshape to wide form."""
    grouped = (
        country_ratios_df_subset.groupby([*group_cols, "Status"])["LengthMergedKmByCountry"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    )
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


km_by_country = pivot_km(["Country"], country_list)
km_by_country.index.name = "Country"

km_by_region = pivot_km(["Region", "SubRegion"], multiindex_region_subregion)
km_by_region.index.names = ["Region", "Subregion"]

km_by_country.loc["Total"] = km_by_country.sum(axis=0).values
# full 2-level key — a flat "Total" label would collapse the MultiIndex to tuples
km_by_region.loc[("Total", ""), :] = km_by_region.sum(axis=0).values

# drop countries with no km in any status, then blank out zeros for display
km_by_country = km_by_country.loc[~(km_by_country == 0).all(axis=1)]
km_by_country = km_by_country.replace(0, "")
km_by_region = km_by_region.replace(0, "")

km_by_region.to_excel(excel_writer, sheet_name="Kilometers by region")
km_by_country.to_excel(excel_writer, sheet_name="Kilometers by country")
km_by_region

In [ ]:
# === consistency guard: pipes tab vs country-ratios tab =====================
# The km tables sum LengthMergedKmByCountry from the ratios tab; the quick
# stats and the release downloads sum LengthMergedKm from the pipes tab. The
# two drift when the backend hasn't refreshed one of them — fail loudly
# instead of shipping a silent gap.
GUARD_TOL_KM = 25

guard = pd.DataFrame({
    "pipes_km": pipes_df_touse.groupby("Status")["LengthMergedKm"].sum(),
    "ratios_km": country_ratios_df_subset.groupby("Status")["LengthMergedKmByCountry"].sum(),
}).reindex(STATUS_LIST).fillna(0)
guard["diff_km"] = guard["pipes_km"] - guard["ratios_km"]
print(guard.round(1).to_string())

bad = guard.loc[guard["diff_km"].abs() > GUARD_TOL_KM]
if not bad.empty:
    # name the offenders so the failure is actionable; grouping by
    # (Wiki, PipelineName) keeps empty-Wiki pipelines from collapsing
    # into one anonymous group
    keys = ["Wiki", "PipelineName", "Status"]
    # fillna on Wiki: aligning two indexes that mix str and NaN is unorderable
    lhs = pipes_df_touse.assign(Wiki=lambda d: d["Wiki"].fillna(""))
    rhs = country_ratios_df_subset.assign(Wiki=lambda d: d["Wiki"].fillna(""))
    offenders = (
        lhs.groupby(keys, dropna=False)["LengthMergedKm"].sum()
        .subtract(
            rhs.groupby(keys, dropna=False)["LengthMergedKmByCountry"].sum(),
            fill_value=0,
        )
    )
    offenders = offenders.loc[offenders.abs() > 1].sort_values(key=abs, ascending=False)
    print("\nper-pipeline mismatches (>1 km), pipes minus ratios:")
    for (wiki, name, status), diff in offenders.head(20).items():
        print(f"  {diff:>+9,.1f} km  {status:<13} {name}")
    if REGION_NAME == "Global":
        raise AssertionError(
            f"pipes and ratios tabs disagree by >{GUARD_TOL_KM} km for {list(bad.index)} "
            "— backend ratios refresh needed before release"
        )
    print(f"\nREGION_NAME={REGION_NAME!r}: totals can differ legitimately — pipes keeps "
          "whole multi-country pipelines, ratios keeps only in-region country rows")

## km by parent company

Each pipeline's `Parent` cell looks like `"TC Energy Corp [60%]; Sempra Energy [40%]"`.
Split into one row per parent, with the bracketed percentages parsed into
`FractionOwnership`.

In [ ]:
CJK_RE = re.compile(r"[\u4e00-\u9fff]+")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?%")
BRACKETS_RE = re.compile(r" \[.*?\]")


def parse_parent_string(parent_string) -> tuple[list[str], list[float]]:
    """Split a Parent cell into ([owner, ...], [fraction, ...]). Empty/NaN parent
    returns (["unknown"], [1.0]). If a percentage is missing, the leftover fraction is
    divided evenly across the owners without one."""
    if parent_string is None or (isinstance(parent_string, float) and np.isnan(parent_string)):
        return ["unknown"], [1.0]
    parent_string = str(parent_string).strip()
    if not parent_string:
        return ["unknown"], [1.0]

    # non-researched QCC owner: leading CJK character with a [100.00%] suffix; keep verbatim
    if CJK_RE.match(parent_string[:1]) and parent_string.endswith("[100.00%]"):
        return [parent_string.removesuffix(" [100.00%]")], [1.0]

    parents = BRACKETS_RE.sub("", parent_string).split("; ")
    pcts = [float(m.rstrip("%")) / 100.0 for m in PERCENT_RE.findall(parent_string)]

    if len(parents) != len(pcts):
        if not pcts:
            pcts = [1.0 / len(parents)] * len(parents)
        else:
            n_missing = len(parents) - len(pcts)
            leftover = 1.0 - float(np.nansum(pcts))
            pcts = pcts + [leftover / n_missing] * n_missing
    return parents, pcts

In [ ]:
# guard: every row in our subset must have a ProjectID
missing_pid = country_ratios_df_subset["ProjectID"].isna()
if missing_pid.any():
    raise ValueError(
        f"missing ProjectID in rows: {country_ratios_df_subset.index[missing_pid].tolist()}"
    )

# build owner-parent rows as a list of dicts, then one DataFrame at the end
# (avoids O(n^2) repeated pd.concat in a loop)
records = []
for row in country_ratios_df_subset.itertuples(index=False):
    parents, pcts = parse_parent_string(row.Parent)
    for parent, frac in zip(parents, pcts):
        records.append(
            {
                "Parent": parent,
                "ProjectID": row.ProjectID,
                "FractionOwnership": frac,
                "Country": row.Country,
                "Status": row.Status,
                "LengthMergedKmByCountry": row.LengthMergedKmByCountry,
            }
        )

owner_parent_calculations_df = pd.DataFrame.from_records(records)
owner_parent_calculations_df["KmOwnership"] = (
    owner_parent_calculations_df["FractionOwnership"]
    * owner_parent_calculations_df["LengthMergedKmByCountry"]
)
owner_parent_calculations_df

In [ ]:
owners_km_by_status_df = (
    owner_parent_calculations_df.groupby(["Parent", "Status"])["KmOwnership"]
    .sum()
    .unstack("Status")
    .reindex(columns=STATUS_LIST)
)
owners_km_by_status_df[IN_DEV_COL] = owners_km_by_status_df[["proposed", "construction"]].sum(axis=1)
owners_km_by_status_df = owners_km_by_status_df[EXCEL_STATUS_LIST]

owners_km_by_status_df.loc["Total"] = owners_km_by_status_df.sum(axis=0, min_count=0).values

owners_km_by_status_df = owners_km_by_status_df.replace({np.nan: "", 0: ""})
owners_km_by_status_df.to_excel(excel_writer, sheet_name="Kilometers by owner")
owners_km_by_status_df

## km by start year, status

In [ ]:
def km_by_year(status_values, year_col: str) -> pd.Series:
    """Sum LengthMergedKm for pipes with the given status(es), grouped by `year_col`.
    `pipes_df_touse` is already subset to the chosen fuel bucket at load time."""
    if isinstance(status_values, str):
        status_values = [status_values]
    subset = pipes_df_touse.loc[
        pipes_df_touse["Status"].isin(status_values)
        & pipes_df_touse[year_col].notna()  # drop missing-year rows so they don't form a NaN group
    ]
    # year cols were coerced to float at load time; cast index back to int for clean display
    grouped = subset.groupby(year_col)["LengthMergedKm"].sum()
    grouped.index = grouped.index.astype(int)
    return grouped


pipes_started_sum      = km_by_year("operating",    "StartYearEarliest")
pipes_construction_sum = km_by_year("construction", "ConstructionYear")
pipes_proposed_sum     = km_by_year("proposed",     "ProposalYear")

# derive the year range from the data, with a floor at 1980 and a ceiling at the
# release year
all_years = pd.concat([pipes_started_sum, pipes_construction_sum, pipes_proposed_sum]).index
YEAR_CEILING = 2026
year_min = int(min(1980, all_years.min())) if len(all_years) else 1980
year_max = int(max(YEAR_CEILING, all_years.max())) if len(all_years) else YEAR_CEILING
year_index = pd.Index(range(year_min, year_max + 1), name="Start year")

km_by_start_year = pd.DataFrame(index=year_index)
km_by_start_year[f"{FUEL_LABEL} pipeline km operating"]    = pipes_started_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km construction"] = pipes_construction_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km proposed"]     = pipes_proposed_sum
km_by_start_year = km_by_start_year.fillna(0)
km_by_start_year.loc["Total"] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, sheet_name="Kilometers by start year")
km_by_start_year.tail(12)

## cost estimates

Builds two artifacts. First, regional and subregional mean `CostUSDPerKm`, drawn from the
**global** set of pipelines (by country-fraction) that have both a known length and a known
per-km cost, trimmed to the [2.5%, 97.5%] interquantile range of `CostUSDPerKm` to drop
extreme outliers. The means are computed globally on purpose — narrowing to a single
`REGION_NAME` would leave too few datapoints in each subregion.

Sparse-sample fallbacks. **A cost average needs at least 3 unique pipelines behind it**
(standing rule, set 2026-09-02); thinner samples inherit the tier above instead of
publishing their own mean:

- a region with fewer than `MIN_REGION_DATAPOINTS` (3) unique pipelines uses the **global** mean
- a subregion with fewer than `MIN_SUBREGION_DATAPOINTS` (3) unique pipelines uses its
  (possibly already-globalized) region's mean

Nov 2025 had no such rule and published Melanesia at US$0.48M/km off a single pipeline —
against Oceania's US$2.68M/km. Under this rule Melanesia inherits Oceania. On both the
Nov 2025 and the current backend no subregion sits in the 3–4 datapoint band, so the
threshold binds only on the 0–1 datapoint subregions (Melanesia, Micronesia, Polynesia).

Second, per-pipeline `CostUSDEstimate = LengthKnownKmByCountry × subregion_mean_cost_per_km`,
overwritten by the row's actual `LengthKnownKmByCountry × CostUSDPerKm` wherever both are
populated. Aggregated into capex (USD billions) by status × country and status × region.

**The capex pivot is fuel-filtered here.** Published GGIT gas capex tabs through Nov 2025
were not — they summed oil, NGL, LPG, CO2 and naphtha rows into the gas table, because the
pivot ran on the unfiltered ratios frame. See this folder's `README.md` for the size of the
effect.

In [ ]:
# Cost-per-km region/subregion means are built from the full global fuel set so that
# small REGION_NAME filters don't shrink the sample size — only the final capex pivot is
# restricted to the chosen region.
country_ratios_fuel_df = country_ratios_df.loc[country_ratios_df["Fuel"].isin(FUEL_OPTIONS)]

# pipes_df_orig is already restricted to FUEL_OPTIONS at load time, so this quantile
# window is the fuel's own cost distribution
cost_df = pipes_df_orig.loc[pipes_df_orig["CostUSDPerKm"].notna()]
if cost_df.empty:
    raise RuntimeError(
        f"no {FUEL_LABEL} pipelines have a numeric CostUSDPerKm — cost estimates cannot run"
    )

q_lo, q_hi = cost_df["CostUSDPerKm"].quantile([0.025, 0.975])
print(f"CostUSDPerKm 2.5% / 97.5% quantiles: {q_lo:,.0f} / {q_hi:,.0f}  (n={len(cost_df):,})")

# country-ratio rows that contribute to the regional means: need both a known
# CostUSDPerKm and a known length, with cost inside the trimmed window
ratios_with_cost = country_ratios_fuel_df.loc[
    country_ratios_fuel_df["CostUSDPerKm"].notna()
    & country_ratios_fuel_df["LengthKnownKmByCountry"].notna()
    & country_ratios_fuel_df["CostUSDPerKm"].between(q_lo, q_hi, inclusive="neither")
]
print(f"country-ratio rows feeding the cost averages: {len(ratios_with_cost):,}")

# global region/subregion lists & subregion->region lookup come from the full country
# dictionary — the regional means are global; the REGION_NAME filter is applied later
region_dict_clean = region_df_orig.loc[
    (region_df_orig["Region"] != "--") & (region_df_orig["SubRegion"] != "--")
]
region_list_global = sorted(region_dict_clean["Region"].dropna().unique())
subregion_list_global = sorted(region_dict_clean["SubRegion"].dropna().unique())
dict_subregion_region = dict(
    zip(region_dict_clean["SubRegion"], region_dict_clean["Region"])
)


def cost_table(level_col: str, level_values: list[str]) -> pd.DataFrame:
    """Mean CostUSDPerKm and unique-ProjectID count, grouped by `level_col`
    (Region or SubRegion)."""
    grouped = ratios_with_cost.groupby(level_col)
    out = pd.DataFrame(
        index=pd.Index(level_values, name=level_col),
        columns=["CostUSDPerKm", "DataPoints"],
        dtype=float,
    )
    out["CostUSDPerKm"] = grouped["CostUSDPerKm"].mean()
    out["DataPoints"] = grouped["ProjectID"].nunique()
    out["DataPoints"] = out["DataPoints"].fillna(0).astype(int)
    return out


pipes_costs_region_df = cost_table("Region", region_list_global)
pipes_costs_subregion_df = cost_table("SubRegion", subregion_list_global)

# Sparse-sample fallbacks. A region with too few datapoints is replaced by the global
# mean; a subregion with too few datapoints is replaced by its (possibly-now-global)
# region mean. The thresholds are inclusive — DataPoints < MIN_* triggers the fallback.
#
# Standing rule (set 2026-09-02): a published cost average needs at least 3 unique
# pipelines behind it. Below that the sample is too thin to defend, so the level inherits
# the tier above rather than publishing its own mean. The case that prompted the rule:
# Nov 2025 released Melanesia at US$0.48M/km off a single pipeline, against Oceania's
# US$2.68M/km. Keep these two in step — they are one rule, split only by tier.
MIN_REGION_DATAPOINTS = 3
MIN_SUBREGION_DATAPOINTS = 3
global_mean_cost = ratios_with_cost["CostUSDPerKm"].mean()
print(f"global mean CostUSDPerKm (post-trim): {global_mean_cost:,.0f}")

sparse_region_mask = (
    pipes_costs_region_df["DataPoints"] < MIN_REGION_DATAPOINTS
) | pipes_costs_region_df["CostUSDPerKm"].isna()
if sparse_region_mask.any():
    print(
        f"replacing region mean with global for {sparse_region_mask.sum()} region(s) "
        f"with < {MIN_REGION_DATAPOINTS} datapoints: "
        f"{pipes_costs_region_df.index[sparse_region_mask].tolist()}"
    )
pipes_costs_region_df.loc[sparse_region_mask, "CostUSDPerKm"] = global_mean_cost

sparse_subregion_mask = (
    pipes_costs_subregion_df["DataPoints"] < MIN_SUBREGION_DATAPOINTS
) | pipes_costs_subregion_df["CostUSDPerKm"].isna()
if sparse_subregion_mask.any():
    # name them: this print is the audit trail for the >= 3 datapoint rule
    thin = pipes_costs_subregion_df.loc[sparse_subregion_mask, "DataPoints"]
    print(
        f"replacing subregion mean with region mean for {sparse_subregion_mask.sum()} "
        f"subregion(s) with < {MIN_SUBREGION_DATAPOINTS} datapoints: "
        + ", ".join(f"{sr} (n={n})" for sr, n in thin.items())
    )
for sr in pipes_costs_subregion_df.index[sparse_subregion_mask]:
    pipes_costs_subregion_df.loc[sr, "CostUSDPerKm"] = pipes_costs_region_df.loc[
        dict_subregion_region[sr], "CostUSDPerKm"
    ]

# write per-km tables to Excel in USD millions per km for readability
(pipes_costs_region_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by region"))
(pipes_costs_subregion_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by subregion"))

pipes_costs_subregion_df.sort_values("CostUSDPerKm", ascending=False)

In [ ]:
# Per-pipeline cost estimate, then capex pivot for the selected REGION_NAME / FUEL bucket.
# Default estimate: known-length x subregion-mean per-km cost.
# Overwrite: rows that already have BOTH a known length and a known per-km cost on the
# pipeline itself get the actual product.
country_ratios_with_capex = country_ratios_df_subset.copy()
country_ratios_with_capex["CostUSDEstimate"] = (
    country_ratios_with_capex["LengthKnownKmByCountry"]
    * country_ratios_with_capex["SubRegion"].map(pipes_costs_subregion_df["CostUSDPerKm"])
)
known_cost_mask = (
    country_ratios_with_capex["LengthKnownKmByCountry"].notna()
    & country_ratios_with_capex["CostUSDPerKm"].notna()
)
country_ratios_with_capex.loc[known_cost_mask, "CostUSDEstimate"] = (
    country_ratios_with_capex.loc[known_cost_mask, "LengthKnownKmByCountry"]
    * country_ratios_with_capex.loc[known_cost_mask, "CostUSDPerKm"]
)


def pivot_capex(group_cols, index) -> pd.DataFrame:
    """Sum CostUSDEstimate by Status × group_cols, reshape to wide form, in USD billions."""
    grouped = (
        country_ratios_with_capex.groupby([*group_cols, "Status"])["CostUSDEstimate"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    ) / 1e9
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


capex_by_country = pivot_capex(["Country"], country_list)
capex_by_country.index.name = "Country"

capex_by_region = pivot_capex(["Region", "SubRegion"], multiindex_region_subregion)
capex_by_region.index.names = ["Region", "Subregion"]

capex_by_country.loc["Total"] = capex_by_country.sum(axis=0).values
# full 2-level key — a flat "Total" label would collapse the MultiIndex to tuples
capex_by_region.loc[("Total", ""), :] = capex_by_region.sum(axis=0).values

# drop countries with no capex in any status, then blank out zeros for display
capex_by_country = capex_by_country.loc[~(capex_by_country == 0).all(axis=1)]
capex_by_country = capex_by_country.replace(0, "")
capex_by_region = capex_by_region.replace(0, "")

capex_by_region.to_excel(excel_writer, sheet_name="Capex USD billions by region")
capex_by_country.to_excel(excel_writer, sheet_name="Capex USD billions by country")
capex_by_region

## comparison against the previously published capex tabs

The fuel filter above is the one deliberate methodology change from the Nov 2025 release.
This cell prints both numbers so the difference is on the record for the release notes,
and so nobody has to rediscover it. Set `SHOW_UNFILTERED_CAPEX = False` to skip.

In [ ]:
SHOW_UNFILTERED_CAPEX = True

if SHOW_UNFILTERED_CAPEX:
    # Reproduce the pre-2026 behaviour: same cost model, but the capex pivot runs on
    # every fuel in the ratios tab rather than the selected bucket.
    unfiltered = country_ratios_df_touse.copy()
    unfiltered["CostUSDEstimate"] = (
        unfiltered["LengthKnownKmByCountry"]
        * unfiltered["SubRegion"].map(pipes_costs_subregion_df["CostUSDPerKm"])
    )
    m = unfiltered["LengthKnownKmByCountry"].notna() & unfiltered["CostUSDPerKm"].notna()
    unfiltered.loc[m, "CostUSDEstimate"] = (
        unfiltered.loc[m, "LengthKnownKmByCountry"] * unfiltered.loc[m, "CostUSDPerKm"]
    )

    def in_dev_total(df):
        d = df.loc[df["Status"].isin(["proposed", "construction"])]
        return d["CostUSDEstimate"].sum() / 1e9

    filt, unfilt = in_dev_total(country_ratios_with_capex), in_dev_total(unfiltered)
    print(f"in-development capex, US$ bn ({REGION_NAME}, {FUEL_LABEL}):")
    print(f"  fuel-filtered (this notebook):        {filt:>8,.1f}")
    print(f"  unfiltered (pre-2026 published):      {unfilt:>8,.1f}")
    print(f"  non-{FUEL_LABEL.lower()} capex previously included: {unfilt - filt:>8,.1f}")

    print(f"\nby fuel, in-development capex US$ bn:")
    by_fuel = (
        unfiltered.loc[unfiltered["Status"].isin(["proposed", "construction"])]
        .groupby("Fuel")["CostUSDEstimate"].sum().div(1e9).sort_values(ascending=False)
    )
    print(by_fuel.round(2).to_string())

## save Excel

In [ ]:
excel_writer.close()
print(f"wrote {output_path}")

## landing-page stats

In [ ]:
# pipes_df_touse is already subset to the chosen fuel bucket and region
print(f"{len(pipes_df_touse):>6,d} {FUEL_LABEL} pipeline projects tracked")
print(f"{pipes_df_touse['LengthMergedKm'].sum() / 1e6:>6.3f} M km tracked")

In [ ]:
in_dev = pipes_df_touse.loc[pipes_df_touse["Status"].isin(["proposed", "construction"])]
print(f"{len(in_dev):>6,d} {FUEL_LABEL} pipeline projects in development (proposed + construction)")
print(f"{in_dev['LengthMergedKm'].sum() / 1e3:>6.1f} K km in development")